# LAB 3: Busca Híbrida em Corpus Jurídico (TCU 2026)
## Avaliação Comparativa BM25 vs kNN (BGE-M3) vs Hybrid RRF vs Hybrid Min-Max — com LangFuse

**Objetivo:** Avaliar a efetividade de quatro estratégias de busca sobre o corpus jurídico enriquecido (`corpus_juridico_aula4_v2.json`, 1.100 acórdãos TCU 2026) usando **MRR@10, Recall@10, NDCG@10** e registrar todas as execuções no **LangFuse** para análise comparativa.

**Pré-requisitos:**
- LAB1 executado (índice `corpus_juridico_aula4` populado, `model_id` Ollama BGE-M3 registrado)
- LAB2 executado (pipelines `hybrid_rrf_pipeline` e `hybrid_minmax_pipeline` criados)
- `lab1_config.json` disponível
- `aula4/datasets/queries_avaliacao_aula4.json` (20 queries com ground-truth)
- LangFuse acessível (`LANGFUSE_HOST`, `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY` em `.env`)

## 1. Setup

In [ ]:
import subprocess, sys
for pkg in ['opensearch-py>=2.7', 'pandas', 'matplotlib', 'seaborn',
            'langfuse>=2.36.0', 'python-dotenv']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

import json, os, time, warnings
from opensearchpy import OpenSearch
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from dotenv import load_dotenv
warnings.filterwarnings('ignore')
load_dotenv()

OPENSEARCH_HOST = os.getenv('OPENSEARCH_HOST', 'localhost')
OPENSEARCH_PORT = int(os.getenv('OPENSEARCH_PORT', 9200))
OPENSEARCH_USER = os.getenv('OPENSEARCH_USER', 'admin')
OPENSEARCH_PASS = os.getenv('OPENSEARCH_PASS', 'admin')

client = OpenSearch(
    hosts=[{'host': OPENSEARCH_HOST, 'port': OPENSEARCH_PORT}],
    http_auth=(OPENSEARCH_USER, OPENSEARCH_PASS),
    use_ssl=False, verify_certs=False, ssl_show_warn=False,
)
print(f"OpenSearch {client.info()['version']['number']} OK")

In [ ]:
with open('lab1_config.json', 'r', encoding='utf-8') as f:
    cfg = json.load(f)
INDEX_NAME = cfg['index_name']
MODEL_ID   = cfg['model_id']
EMBED_DIM  = cfg['embed_dim']
PIPELINE_RRF    = 'hybrid_rrf_pipeline'
PIPELINE_MINMAX = 'hybrid_minmax_pipeline'
print(f'Índice: {INDEX_NAME} | model_id: {MODEL_ID} | dim: {EMBED_DIM}')
print(f'Pipelines: {PIPELINE_RRF}, {PIPELINE_MINMAX}')

## 2. LangFuse — Instrumentação

In [ ]:
from langfuse import Langfuse

LANGFUSE_HOST       = os.getenv('LANGFUSE_HOST', 'http://localhost:3000')
LANGFUSE_PUBLIC_KEY = os.getenv('LANGFUSE_PUBLIC_KEY', 'pk-lf-xxxx')
LANGFUSE_SECRET_KEY = os.getenv('LANGFUSE_SECRET_KEY', 'sk-lf-xxxx')

try:
    lf = Langfuse(
        public_key=LANGFUSE_PUBLIC_KEY,
        secret_key=LANGFUSE_SECRET_KEY,
        host=LANGFUSE_HOST,
    )
    LANGFUSE_OK = True
    print(f'LangFuse OK em {LANGFUSE_HOST}')
except Exception as e:
    LANGFUSE_OK = False
    lf = None
    print(f'LangFuse indisponível ({e}). Métricas só serão impressas.')

## 3. Carregar Queries de Avaliação (TCU 2026)

20 queries com ground-truth gerado a partir do conteúdo real dos acórdãos.

In [ ]:
with open('../datasets/queries_avaliacao_aula4.json', 'r', encoding='utf-8') as f:
    queries = json.load(f)
print(f'Queries: {len(queries)}')
for q in queries[:3]:
    print(f"  [{q['id']}] {q['texto'][:60]}... -> {len(q['documentos_relevantes'])} relevantes")

## 4. Métricas de IR (MRR, Recall, NDCG)

In [ ]:
def calc_mrr(results, relevant, k=10):
    for rank, d in enumerate(results[:k], 1):
        if d in relevant:
            return 1.0 / rank
    return 0.0

def calc_recall(results, relevant, k=10):
    if not relevant:
        return 0.0
    return len(set(results[:k]) & set(relevant)) / len(relevant)

def calc_ndcg(results, relevant, k=10):
    dcg = sum((1.0 if d in relevant else 0.0) / np.log2(i + 1)
              for i, d in enumerate(results[:k], 1))
    idcg = sum(1.0 / np.log2(i + 1) for i in range(1, min(len(relevant), k) + 1))
    return dcg / idcg if idcg > 0 else 0.0

print('Métricas definidas.')

## 5. Funções de Busca (server-side, sem embeddings no cliente)

In [ ]:
K = 10

def _ids(hits):
    return [h['_source']['id'] for h in hits]

def search_bm25(q, k=K):
    t0 = time.perf_counter()
    r = client.search(index=INDEX_NAME,
                      body={'size': k,
                            'query': {'match': {'conteudo': {'query': q}}},
                            '_source': ['id']})
    return _ids(r['hits']['hits']), (time.perf_counter() - t0) * 1000

def search_neural(q, k=K):
    t0 = time.perf_counter()
    r = client.search(index=INDEX_NAME,
                      body={'size': k,
                            'query': {'neural': {'knn_vector': {
                                'query_text': q, 'model_id': MODEL_ID, 'k': k}}},
                            '_source': ['id']})
    return _ids(r['hits']['hits']), (time.perf_counter() - t0) * 1000

def search_hybrid(q, pipeline, k=K):
    t0 = time.perf_counter()
    body = {'size': k,
            'query': {'hybrid': {'queries': [
                {'match':  {'conteudo': {'query': q}}},
                {'neural': {'knn_vector': {'query_text': q, 'model_id': MODEL_ID, 'k': k}}},
            ]}},
            '_source': ['id']}
    r = client.search(index=INDEX_NAME, body=body,
                       params={'search_pipeline': pipeline})
    return _ids(r['hits']['hits']), (time.perf_counter() - t0) * 1000

estrategias = {
    'BM25':          lambda q: search_bm25(q),
    'kNN_BGE_M3':    lambda q: search_neural(q),
    'Hybrid_RRF':    lambda q: search_hybrid(q, PIPELINE_RRF),
    'Hybrid_MinMax': lambda q: search_hybrid(q, PIPELINE_MINMAX),
}
print('Funções de busca definidas:', list(estrategias))

## 6. Avaliação Completa + Registro no LangFuse

In [ ]:
if LANGFUSE_OK:
    trace = lf.trace(
        name='Aula4-LAB3-Hybrid-Search-Juridico',
        metadata={
            'aula': '4', 'lab': '3',
            'index': INDEX_NAME, 'embed_model': cfg.get('embed_model', 'bge-m3'),
            'embed_dim': EMBED_DIM, 'n_queries': len(queries), 'k': K,
            'corpus': 'TCU 2026 — 1.100 acórdãos'
        }
    )
else:
    trace = None

resultados = []
for q in queries:
    relev = q['documentos_relevantes']
    for nome, fn in estrategias.items():
        ids, lat_ms = fn(q['texto'])
        mrr    = calc_mrr(ids, relev, k=K)
        recall = calc_recall(ids, relev, k=K)
        ndcg   = calc_ndcg(ids, relev, k=K)

        resultados.append({
            'query_id': q['id'], 'estrategia': nome,
            'MRR@10': mrr, 'Recall@10': recall, 'NDCG@10': ndcg,
            'latencia_ms': lat_ms,
        })

        if trace is not None:
            sp = trace.span(
                name=f"{q['id']}-{nome}",
                metadata={'query': q['texto'], 'k': K, 'top_ids': ids}
            )
            sp.score(name='MRR_10',     value=mrr)
            sp.score(name='Recall_10',  value=recall)
            sp.score(name='NDCG_10',    value=ndcg)
            sp.score(name='latency_ms', value=lat_ms)
            sp.end()

df = pd.DataFrame(resultados)
if trace is not None:
    lf.flush()
    print(f'\nTrace LangFuse: {LANGFUSE_HOST}/traces/{trace.id}')
print(f'\nAvaliação completa: {len(df)} combinações query×modo')
df.head(8)

## 7. Resumo Agregado por Estratégia

In [ ]:
resumo = (
    df.groupby('estrategia')[['MRR@10', 'Recall@10', 'NDCG@10', 'latencia_ms']]
      .agg(['mean', 'std']).round(4)
)
print('Resumo agregado por estratégia:')
print(resumo)

print('\nRanking por MRR@10 médio:')
print(df.groupby('estrategia')['MRR@10'].mean().sort_values(ascending=False).round(4))

# Registrar agregados no LangFuse
if trace is not None:
    for nome, grp in df.groupby('estrategia'):
        for m in ['MRR@10', 'Recall@10', 'NDCG@10', 'latencia_ms']:
            trace.score(name=f'{m}_mean_{nome}', value=float(grp[m].mean()),
                        comment=f'Média {m} para {nome}')
    lf.flush()
    print(f'\nScores agregados registrados em {LANGFUSE_HOST}/traces/{trace.id}')

## 8. Visualizações

In [ ]:
# Bar plot: MRR@10
fig, ax = plt.subplots(figsize=(10, 5))
ordem = df.groupby('estrategia')['MRR@10'].mean().sort_values(ascending=False).index.tolist()
sns.barplot(data=df, x='estrategia', y='MRR@10', order=ordem,
            palette='viridis', ax=ax, errorbar='sd')
ax.set_title('MRR@10 por Estratégia — Corpus TCU 2026', fontweight='bold')
ax.set_ylim(0, 1.05); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig('lab3_mrr.png', dpi=140); plt.show()

In [ ]:
# Heatmap Recall@10 por query×estratégia
pivot = df.pivot(index='query_id', columns='estrategia', values='Recall@10')
plt.figure(figsize=(9, 8))
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='YlGnBu',
            cbar_kws={'label': 'Recall@10'}, linewidths=.5)
plt.title('Recall@10 por Query × Estratégia', fontweight='bold')
plt.tight_layout(); plt.savefig('lab3_recall_heatmap.png', dpi=140); plt.show()

In [ ]:
# Trade-off: Latência vs MRR
fig, ax = plt.subplots(figsize=(8, 5))
agg = df.groupby('estrategia').agg(MRR=('MRR@10', 'mean'),
                                    lat=('latencia_ms', 'mean')).reset_index()
ax.scatter(agg['lat'], agg['MRR'], s=180, c=range(len(agg)),
           cmap='tab10', edgecolors='black')
for _, r in agg.iterrows():
    ax.annotate(r['estrategia'], (r['lat'], r['MRR']),
                xytext=(7, 7), textcoords='offset points', fontsize=10)
ax.set_xlabel('Latência média (ms)'); ax.set_ylabel('MRR@10')
ax.set_title('Trade-off Qualidade vs Latência', fontweight='bold')
ax.grid(True, alpha=0.3); plt.tight_layout(); plt.savefig('lab3_tradeoff.png', dpi=140); plt.show()

## 9. Análise Qualitativa — 3 Queries

In [ ]:
for qid in ['Q01', 'Q05', 'Q11']:
    q = next(x for x in queries if x['id'] == qid)
    print(f"\n=== {qid}: {q['texto']} ===")
    row = df[df['query_id'] == qid]
    print(row[['estrategia', 'MRR@10', 'Recall@10', 'NDCG@10', 'latencia_ms']]
          .round(4).to_string(index=False))

## 10. Exportar Resultados

Salvo em CSV para reuso no LAB6 (dashboard consolidado).

In [ ]:
df.to_csv('lab3_resultados.csv', index=False)
print('lab3_resultados.csv gravado.')
print(df.groupby('estrategia')[['MRR@10', 'Recall@10', 'NDCG@10']].mean().round(4))

## Conclusões esperadas

Em corpus jurídico com terminologia técnica + paráfrases:

- **BM25 sozinho**: bom em queries com termos exatos (números de leis, palavras-chave) — falha em sinônimos.
- **kNN BGE-M3**: forte em paráfrases e queries semânticas — pode trazer documentos tematicamente próximos sem o termo exato.
- **Hybrid RRF**: melhor MRR/NDCG global; default seguro.
- **Hybrid Min-Max**: depende dos pesos; tunável para casos de uso específicos.

## Referências (ABNT)

CORMACK, G. V.; CLARKE, C. L. A.; BUETTCHER, S. **Reciprocal Rank Fusion outperforms Condorcet**. SIGIR, 2009.

OPENSEARCH PROJECT. **Hybrid Search**. 3.0 Docs. <https://docs.opensearch.org/3.0/vector-search/ai-search/hybrid-search/>.

CHEN, J. et al. **BGE M3-Embedding**. arXiv:2309.07597, 2024.

LANGFUSE. **Scores API**. Docs. <https://langfuse.com/docs/scores>.

TRIBUNAL DE CONTAS DA UNIÃO. **Acórdãos 2026 — base completa**.